# 가까운 학교 기록 테이블 전처리 수행

In [2]:
import numpy as np
import pandas as pd

In [3]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [4]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_nearbyschool`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

       id  distance  nearby_school_id  school_id
0  119021  0.004564                 6          7
1  119022  0.010787                13          7
2  119023  0.012928                20          7
3  119024  0.013590                 4          7
4  119025  0.014122                24          7


## 결측치 확인 및 데이터 정보 확인

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59500 entries, 0 to 59499
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                59500 non-null  Int64  
 1   distance          59500 non-null  float64
 2   nearby_school_id  59500 non-null  Int64  
 3   school_id         59500 non-null  Int64  
dtypes: Int64(3), float64(1)
memory usage: 2.0 MB


* 결측치(Missing Value) 없음

* 전체 컬럼 수치형(Numeric) 데이터로 구성

## 중복값 체크

In [6]:
# 중복값 체크 
df.duplicated().sum()

df['id'].duplicated().sum()

# 학교, 학년, 반 중복 확인
df[df[['nearby_school_id', 'school_id']].duplicated(keep=False)]

,id,distance,nearby_school_id,school_id


* 레코드 유일성: 전체 행 기준 완전 중복 및 고유 식별자(id) 중복 없음 확인 (0건)

* 학교 간 관계 매핑 중복: 학교(school_id)와 근접 학교(nearby_school_id) 쌍(Pair) 기준 중복 레코드 부재 확인

## 이상치 판별

In [7]:
# 이상치 판별
df.describe()

,id,distance,nearby_school_id,school_id
count,59500.0,59500.000000,59500.0,59500.0
mean,148770.5,0.055338,2976.695882,2980.524538
std,17176.314845,0.604024,1706.965231,1718.663078
min,119021.0,0.000000,4.0,4.0
25%,133895.75,0.011484,1518.0,1493.0
50%,148770.5,0.021658,2973.0,2980.5
75%,163645.25,0.064555,4448.0,4469.0
max,178520.0,49.296594,5964.0,5964.0


* 거리(distance) 컬럼의 양극단값 검토 필요:

  * 최솟값(0): 동일 학교 간의 자기 참조(Self-Join)로 인한 것인지 유효성 확인 필요

  * 최댓값(49): 전체 분포 대비 유독 튀는 극단치(Outlier)로 식별됨

  * 단위 기준 정의 필요: 데이터 수집 단위(km, m 등)와 일반적인 근접 학교 반경을 고려할 때 이상 입력값일 가능성이 있어 도메인 기준 검증 필요

### 거리 이상치 확인

In [8]:
# 거리 0인 데이터 확인
df[df['distance'] == 0]['id'].count()

np.int64(6214)

In [9]:
# 거리가 0이지만 본인 학교를 참조한 케이스
df[(df['distance'] == 0) & (df['nearby_school_id'] == df['school_id'])] # 5950건

# 거리가 0이지만 본인 학교가 아닌 케이스 필터링
df[(df['distance'] == 0) & (df['nearby_school_id'] != df['school_id'])] # 264건

,id,distance,nearby_school_id,school_id
220,119241,0.0,28,27
230,119251,0.0,27,28
3170,122191,0.0,324,323
3200,122221,0.0,323,324
3220,122241,0.0,329,328
...,...,...,...,...
55460,174481,0.0,5553,5552
55950,174971,0.0,5619,5598
56120,175141,0.0,5598,5619
57040,176061,0.0,5720,5713


In [10]:
# 1. 거리 0인 비-자기참조 쌍 추출 (264건)
zero_dist_pairs = df[(df['distance'] == 0) & (df['nearby_school_id'] != df['school_id'])]

# A-B, B-A 중복을 없애기 위해 정렬된 튜플 쌍으로 고유 쌍 추출 (총 132쌍 예상)
unique_pairs = set(
    tuple(sorted([row['school_id'], row['nearby_school_id']]))
    for _, row in zero_dist_pairs.iterrows()
)

# 2. 각 학교별 주변 학교 집합(Set) 미리 생성 (단, 자기 자신과 상대방 ID는 비교에서 제외)
school_to_nearby = df.groupby('school_id')['nearby_school_id'].apply(set).to_dict()

# 3. 각 쌍별 주변 학교 목록 일치 여부 및 유사도(Jaccard Similarity) 계산
results = []
for s1, s2 in unique_pairs:
    # 각 학교의 주변 학교 목록에서 서로의 ID와 자기 자신을 제외한 순수 주변 학교 추출
    neighbors1 = school_to_nearby.get(s1, set()) - {s1, s2}
    neighbors2 = school_to_nearby.get(s2, set()) - {s1, s2}
    
    # 교집합과 합집합 계산
    intersection = neighbors1 & neighbors2
    union = neighbors1 | neighbors2
    
    similarity = len(intersection) / len(union) if union else 1.0
    
    results.append({
        'school_A': s1,
        'school_B': s2,
        'A_neighbors_cnt': len(neighbors1),
        'B_neighbors_cnt': len(neighbors2),
        'common_cnt': len(intersection),
        'similarity': round(similarity, 4),
        'is_exact_match': neighbors1 == neighbors2
    })

# 결과 데이터프레임 변환
verification_df = pd.DataFrame(results).sort_values(by='similarity', ascending=False)
display(verification_df.head(15))

# 전체 요약 출력
print(f"총 검증 대상 쌍: {len(verification_df)}쌍")
print(f"주변 학교가 100% 일치하는 쌍: {verification_df['is_exact_match'].sum()}쌍")
print(f"유사도 90% 이상인 쌍: {(verification_df['similarity'] >= 0.9).sum()}쌍")

,school_A,school_B,A_neighbors_cnt,B_neighbors_cnt,common_cnt,similarity,is_exact_match
0,3408,3409,8,8,8,1.0,True
83,5197,5202,8,8,8,1.0,True
97,3051,3053,8,8,8,1.0,True
96,4501,4509,8,8,8,1.0,True
95,5096,5097,8,8,8,1.0,True
94,4497,4509,8,8,8,1.0,True
93,3326,3328,8,8,8,1.0,True
92,4497,4500,8,8,8,1.0,True
91,4536,4537,8,8,8,1.0,True
90,2447,2448,8,8,8,1.0,True


총 검증 대상 쌍: 132쌍
주변 학교가 100% 일치하는 쌍: 132쌍
유사도 90% 이상인 쌍: 132쌍


* 동일 좌표 학교군(132쌍) 식별 및 원인 분석:

  * 식별 현상: 상호 거리(distance = 0.0) 및 주변 학교 네트워크가 완전 일치하는 학교 쌍이 총 264건(132쌍) 확인됨.

  * 도메인 원인: 동일 부지/캠퍼스를 공유하여 지리적 좌표가 일치하는 병설 학교군으로 확인됨.

    * 중·고등학교 병설 사례 (예: 휘문중 - 휘문고)

    * 남·여 고등학교 병설 사례 (예: 세화고 - 세화여고)

* 데이터 정제 및 유지 의사결정:

  * 학교 식별자(ID) 유지: 본 분석의 핵심 목적이 '학교별 주변 영향도 파악'이므로, 물리적 위치가 같더라도 개별 학교 단위의 고유 엔티티(ID)로서의 가치를 인정하여 데이터 통합 없이 유지 결정.

  * 자기 참조 레코드 정제: 불필요한 중복 연산 방지 및 영향도 지표의 순도 확보를 위해 본인을 참조하는 행(`school_id == nearby_school_id`, 0.0km)은 삭제 처리 진행 예정.

In [11]:
# 0부터 최대값(또는 50)까지 1 단위로 구간 생성
bins = np.arange(0, np.ceil(df['distance'].max()) + 1, 1)

# 구간별 카운트 (오름차순 정렬)
dist_distribution = pd.cut(df['distance'], bins=bins, right=False).value_counts().sort_index()

# 0건인 구간 제외하고 데이터가 있는 구간만 출력
print(dist_distribution[dist_distribution > 0])

distance
[0.0, 1.0)      59432
[1.0, 2.0)         50
[6.0, 7.0)          9
[43.0, 44.0)        1
[48.0, 49.0)        2
[49.0, 50.0)        6
Name: count, dtype: int64


* 원거리(distance ≥ 6.0km) 이상치 식별 및 검토:

  * 분포 왜곡 현상: 전체 레코드의 약 99.9%가 2.0km 미만에 집중되어 있는 반면, 6.0km 이상 구간은 총 18건(6~7km: 9건, 43~50km: 9건)으로 표본이 극히 제한적임.

  * 근접 기준 불일치: '근접 학교'라는 정의상 도보/통학 생활권을 벗어난 40km 이상의 극단값은 좌표 결측(대체값)이나 수집 파이프라인 상의 오류일 가능성이 높음.

  * 조치 계획: 해당 원거리 데이터 18건의 소속 학교 및 좌표를 직접 대조하여 유효성 확인 후 필터링 기준선(Cut-off) 수립 예정.

In [12]:
# 6km 이상인 18건의 데이터 상세 확인
outliers = df[df['distance'] >= 6.0].sort_values(by='distance', ascending=False)
display(outliers[['school_id', 'nearby_school_id', 'distance']])

,school_id,nearby_school_id,distance
59308,5948,615,49.296594
59307,5948,618,49.288493
59306,5948,553,49.224064
59305,5948,1155,49.222822
59304,5948,1026,49.193698
59303,5948,1171,49.149385
59302,5948,1420,48.410895
59301,5948,1425,48.403180
59300,5948,5949,43.298250
59358,5949,570,6.254296


* 이상치 발생 군집 식별:

  * 6km 이상 및 40km대에 달하는 극단값(Outlier) 전량이 특정 학교군(school_id: 5948, 5949)에만 국한되어 나타나는 현상 확인.

  * 전체 데이터셋 전반의 분산 오류가 아닌, 해당 두 개 학교의 지리 좌표 수집 누락·오기입 등 개별 엔티티 단위의 수집 파이프라인 결함일 가능성이 높음.

In [13]:
df[df['school_id'].isin([5948, 5949])].count()

id                  20
distance            20
nearby_school_id    20
school_id           20
dtype: int64

In [14]:
df[df['nearby_school_id'].isin([5948, 5949])]

,id,distance,nearby_school_id,school_id
59300,178321,43.29825,5949,5948
59309,178330,0.00000,5948,5948
59359,178380,0.00000,5949,5949


* 이상 학교군(5948, 5949) 전수 오류 진단:

  * 연결 구조 분석: 두 학교 각각 매핑된 근접 학교 레코드는 총 10건(두 학교 합산 20건)이며, 이는 자기 참조 행(distance = 0)을 포함한 전체 연결 데이터에 해당함.

  * 결론 및 판단: 두 학교에 할당된 전체 근접 학교 매핑이 전반적으로 비정상 거리(6km 이상 및 40km대 극단값)를 나타내고 있어, 일부 연결의 문제가 아닌 해당 학교들의 '기준 좌표 자체가 잘못 입력된 원천 데이터 오류'로 최종 판단됨.

In [15]:
temp_sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_school`
    WHERE id IN (5948, 5949)
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

     id address  student_count school_type
0  5949       -              1           H
1  5948       -              2           H


* 식별 및 추적 한계:

  * 해당 이상 학교군(5948, 5949)은 학교 데이터 내 기본 주소 정보가 결측되어 있어, 실제 학교를 특정하거나 정확한 물리적 위치 및 주변 연결 관계를 보정·검증하기 불가능함.

* 데이터 정제(삭제) 조치 결정:

  * 좌표 오류로 인해 연결된 데이터 전반의 신뢰도가 상실되었고 보정이 불가하므로, 분석 왜곡 방지를 위해 해당 두 학교와 관련된 모든 근접 매핑 레코드(총 20건)는 삭제 처리하기로 결정함.

### 역방향 관계 확인 (추가)

In [26]:
# 원본에서 역관계 기준으로 사용할 데이터 생성
reverse_df = df[["school_id", "nearby_school_id", "distance"]].copy()

# school_id <-> nearby_school_id 뒤집기
reverse_df = reverse_df.rename(columns={
    "school_id": "nearby_school_id",
    "nearby_school_id": "school_id"
})

# 현재 행의 역관계 + distance가 존재하는지 확인
check_df = df.merge(
    reverse_df.drop_duplicates(),
    on=["school_id", "nearby_school_id", "distance"],
    how="left",
    indicator=True
)

# 역관계가 없는 행만 추출
missing_reverse = check_df[
    check_df["_merge"] == "left_only"
].drop(columns="_merge")

print(missing_reverse)

           id  distance  nearby_school_id  school_id
22     119043  0.115943                29         22
23     119044  0.127038                36         22
24     119045  0.129498                 5         22
25     119046  0.136586                30         22
27     119048  0.143549               228         22
...       ...       ...               ...        ...
59473  178494  0.053842              4832       5964
59475  178496  0.055266              4514       5964
59476  178497  0.055511              4585       5964
59477  178498  0.058339              4583       5964
59478  178499  0.069071              4844       5964

[13182 rows x 4 columns]


* 역방향 기록이 없는 데이터가 있으므로 확인 진행

* 역방향의 데이터가 없는 13182건 데이터 확인

In [27]:
# 제외 대상 ID 리스트
target_ids = [5948, 5949]

has_target = missing_reverse['nearby_school_id'].isin(target_ids) | missing_reverse['school_id'].isin(target_ids)

# 해당 조건을 만족하지 않는(~) 행만 필터링
result_df = missing_reverse[~has_target]

# 결과 확인
result_df

,id,distance,nearby_school_id,school_id
22,119043,0.115943,29,22
23,119044,0.127038,36,22
24,119045,0.129498,5,22
25,119046,0.136586,30,22
27,119048,0.143549,228,22
...,...,...,...,...
59473,178494,0.053842,4832,5964
59475,178496,0.055266,4514,5964
59476,178497,0.055511,4585,5964
59477,178498,0.058339,4583,5964


* 이상 학교 (5948, 5949)를 제외하더라도 13164건의 데이터가 쌍으로 존재하지는 않고 있다.

In [ ]:
# missing_reverse의 역관계 생성
insert_rows = missing_reverse[
    ["distance", "nearby_school_id", "school_id"]
].copy()

insert_rows = insert_rows.rename(columns={
    "nearby_school_id": "school_id",
    "school_id": "nearby_school_id"
})

# 현재 전체 데이터에서 가장 큰 ID 확인
max_id = df["id"].max()

# 새로운 ID를 max_id + 1부터 순차적으로 부여
insert_rows.insert(
    0,
    "id",
    range(max_id + 1, max_id + 1 + len(insert_rows))
)

# SQL VALUES 생성
values = ",\n".join(
    f"({row.id}, {row.distance}, {row.nearby_school_id}, {row.school_id})"
    for row in insert_rows.itertuples(index=False)
)

query = f"""
INSERT INTO `project.dataset.table`
    (id, distance, nearby_school_id, school_id)
VALUES
    {values};
"""

print(query)



INSERT INTO `project.dataset.table`
    (id, distance, nearby_school_id, school_id)
VALUES
    (178521, 0.1159434513, 22, 29),
(178522, 0.1270377379, 22, 36),
(178523, 0.1294982195, 22, 5),
(178524, 0.136586099, 22, 30),
(178525, 0.1435493618, 22, 228),
(178526, 0.1447461423, 22, 14),
(178527, 0.0235161416, 20, 13),
(178528, 0.1101300364, 16, 18),
(178529, 0.1120838181, 16, 25),
(178530, 0.1162576675, 16, 9),
(178531, 0.119721749, 16, 24),
(178532, 0.122776738, 16, 20),
(178533, 0.1260176673, 16, 7),
(178534, 0.1269404028, 16, 13),
(178535, 0.1301669131, 16, 6),
(178536, 0.101558964, 23, 32),
(178537, 0.1017370541, 23, 17),
(178538, 0.1028027794, 23, 12),
(178539, 0.1057990825, 23, 14),
(178540, 0.1058507401, 23, 8),
(178541, 0.1095095399, 23, 30),
(178542, 0.0204503729, 18, 6),
(178543, 0.0273890803, 18, 4),
(178544, 0.0321815407, 18, 26),
(178545, 0.133870598, 11, 12),
(178546, 0.022239658, 9, 13),
(178547, 0.0276858608, 9, 4),
(178548, 0.031233967, 9, 7),
(178549, 0.0315714979, 9, 

In [20]:
# 거리가 0이면서 자기참조가 되어있는 데이터 삭제
# 5948 학교와 5949 학교의 모든 거리 관계 삭제

# 2. DML 삭제 쿼리 실행
delete_sql = f"""
DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_nearbyschool`
WHERE 
    -- 1) 거리가 0이면서 자기 참조인 데이터 삭제
    (distance = 0 AND school_id = nearby_school_id)
    
    -- 2) 5948, 5949 학교와 관련된 모든 거리 관계 삭제 (기준 학교 및 근접 학교 포함)
    OR (school_id IN (5948, 5949) OR nearby_school_id IN (5948, 5949))
"""

query_job = client.query(delete_sql)
query_job.result()  # DML 완료 대기

print(f"\n삭제 완료! 총 삭제된 행 수: {query_job.num_dml_affected_rows}건")


삭제 완료! 총 삭제된 행 수: 5968건
